In [5]:
import pandas as pd
import psycopg2
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
import numpy as np
from collections import Counter
import time 
import requests 
import pickle
import random 
import matplotlib.pyplot as plt
import glob 

main_path = 'reviews/'

import warnings
warnings.filterwarnings("ignore")

## Novelty

### Get Sam ARTS novelty measure 

In [6]:
# download data from Sam ARTS

import urllib.request
urllib.request.urlretrieve("https://zenodo.org/records/8283353/files/papers_textual_metrics.csv?download=1", main_path + "data_reviews/SAME_ARTS_papers_textual_metrics.csv")

('/home/fs01/spec1142/Emma/GateKeepers/reviews/data_reviews/SAME_ARTS_papers_textual_metrics.csv',
 <http.client.HTTPMessage at 0x7f5325b47670>)

In [13]:
# load file with the data to run the regression at the paper level. 

file_papers = pd.read_stata(main_path + 'data_reviews/science_papers_cites_novelty_05.dta')
list_works = list(file_papers['work_id'])

len(list_works)

21458438

In [14]:
# only select papers from our file from Sam ARTS novelty measures

df_order_authors = pd.read_csv(main_path + "data_reviews/SAME_ARTS_papers_textual_metrics.csv", chunksize = 5000000)

df_works_authors = pd.DataFrame()

for data in tqdm(df_order_authors):
                
    ## load citing data as a dataframe
    df = pd.DataFrame(data , dtype = str)
    df['work_id'] = "W" + df['PaperID']

    ## only keep work id that are in the PQR file
    df = df[df['work_id'].isin(list_works)]
    df_works_authors = pd.concat( [ df_works_authors,df] )
    print(len(df_works_authors))
    


0it [00:00, ?it/s]

953


1it [00:48, 48.37s/it]IOStream.flush timed out


231037


3it [02:23, 47.87s/it]

...

14it [12:20, 58.30s/it]

13508304


15it [12:57, 51.95s/it]

13914184


15it [12:57, 51.86s/it]


In [23]:
# reformat column as integer
df_works_authors['new_word'] = df_works_authors['new_word'].astype(int)


In [16]:
# save Sam ARTS novelty measures for the relevant PQR papers. 
df_works_authors.to_csv(main_path + "data_reviews/SAM_ARTS_novelty_measures_PQRs_papers.tsv" , "\t", index = False)

### Merge with our data 

In [36]:
## load files with PQR data at the paper level

file_papers = pd.read_stata(main_path + 'data_reviews/science_papers_cites_novelty_3yearswindow_05.dta')

In [27]:
## load Sam ARTS measure for the corresponding papers

df_Sam_ARTS = pd.read_csv(main_path + "data_reviews/SAM_ARTS_novelty_measures_PQRs_papers.tsv" ,delimiter = "\t")

In [37]:
## merge the two files 

df_new_novelty = df_Sam_ARTS.merge(file_papers, on='work_id' , how = 'inner')

In [38]:
df_new_novelty.head()

,PaperID,new_word,new_word_reuse,new_bigram,new_bigram_reuse,new_trigram,new_trigram_reuse,new_word_comb,new_word_comb_reuse,cosine_max,...,concept1_int,top_5_percent,check_5percent,company,education,mean_age_backward_cite,mean_diverse_works,count_backward_cites,self_cites,self_cites_prop
0,2134293809,0,0,0,0,0,0,0,0,0.653590,...,90,0,0,0,0,NaN,NaN,NaN,NaN,NaN
1,2072305923,0,0,2,7,3,15,246,946,0.416660,...,26,0,0,0,0,NaN,NaN,NaN,NaN,NaN
2,2071065684,0,0,0,0,0,0,0,0,0.031321,...,48,0,0,0,1,NaN,NaN,NaN,NaN,NaN
3,2749929368,0,0,2,10,0,0,161,805,0.422206,...,104,0,0,0,0,NaN,NaN,NaN,NaN,NaN
4,2042968838,0,0,2,224,0,0,165,585214,0.431785,...,-1,0,0,0,1,NaN,NaN,NaN,NaN,NaN


In [40]:
# only save relevant fields

df_new_novelty = df_new_novelty[['PaperID', 'new_word', 'new_word_reuse', 'new_bigram',
       'new_bigram_reuse', 'new_trigram', 'new_trigram_reuse', 'new_word_comb',
       'new_word_comb_reuse', 'cosine_max', 'cosine_avg', 'n_words',
       'n_bigrams', 'n_trigrams', 'work_id', 
       'mean_age_backward_cite', 'mean_diverse_works', 'count_backward_cites',
       'self_cites', 'index', 'number_only_authors', 'number_active_gks',
       'number_career_gks', 'pubyear', 'cites', 'backward_cites',
       'Computer_science', 'Medicine', 'Biology', 'Physics', 'Chemistry',
       'Materials_science', 'PPP', 'institution_int', 'concept1_int',
       'top_5_percent', 'check_5percent', 'company', 'education',
       'novelty', 'self_cites_prop']]

In [41]:
# save file

df_new_novelty.to_stata(main_path + "data_reviews/science_papers_cites_novelty_05_3years_window_SAM_ARTS_NOVELTY.dta")

## Weak PQRs VS Strong PQRs

In [76]:
## load files 

file_papers = pd.read_stata(main_path + "data_reviews/scientist_years_cites_novelty_backward_cites05.dta")
file_patents = pd.read_stata(main_path + 'data_reviews/inventors_years_cites_novelty_backward_cites05.dta')

In [77]:
## load PQR file 

df_gatekeepers = pd.read_csv(main_path+ "data_reviews/gatekeepers_intermediate_clean_v5.tsv" , delimiter = "\t")

In [78]:
## merge files

file_patents = file_patents.merge(df_gatekeepers[['inventor_id','number of papers']], on = 'inventor_id')
file_papers = file_papers.merge(df_gatekeepers[['author_id','number of patents']], on = 'author_id')

In [81]:
## save files (with number of papers and number of patents)

file_papers.to_stata(main_path + "data_reviews/scientist_years_cites_novelty_backward_cites05_nb_patents.dta")
file_patents.to_stata(main_path + "data_reviews/inventors_years_cites_novelty_backward_cites05_nb_papers.dta")